In [2]:
import os
import zipfile
import urllib.request
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.neighbors import NearestNeighbors

for layer in ["bronze", "silver", "gold"]:
    os.makedirs(layer, exist_ok=True)

DATA_URL = "https://cdn.freecodecamp.org/project-data/books/book-crossings.zip"
ZIP_PATH = "book-crossings.zip"
RAW_DIR = "raw_data"

if not os.path.exists(ZIP_PATH):
    print("Downloading Book-Crossing dataset...")
    req = urllib.request.Request(
        DATA_URL,
        headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/111.0.0.0 Safari/537.36'}
    )
    with urllib.request.urlopen(req) as response, open(ZIP_PATH, 'wb') as out_file:
        out_file.write(response.read())

os.makedirs(RAW_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(RAW_DIR)

print("Extracted files:", os.listdir(RAW_DIR))
print("Medallion directories ready:", [d for d in ["bronze", "silver", "gold"] if os.path.isdir(d)])

Extracted files: ['BX-Book-Ratings.csv', 'BX-Users.csv', 'BX-Books.csv']
Medallion directories ready: ['bronze', 'silver', 'gold']


In [3]:
ingestion_ts = datetime.now(timezone.utc).isoformat()

raw_files = {
    "users": os.path.join(RAW_DIR, "BX-Users.csv"),
    "books": os.path.join(RAW_DIR, "BX-Books.csv"),
    "ratings": os.path.join(RAW_DIR, "BX-Book-Ratings.csv"),
}

for table_name, file_path in raw_files.items():
    df_raw = pd.read_csv(
        file_path,
        sep=";",
        encoding="latin-1",
        escapechar="\\",
        dtype=str,
        on_bad_lines="warn"
    )

    df_raw["ingestion_timestamp"] = ingestion_ts

    out_path = os.path.join("bronze", f"{table_name}.parquet")
    df_raw.to_parquet(out_path, index=False)
    print(f"Saved {out_path} -> shape: {df_raw.shape}")

Saved bronze/users.parquet -> shape: (278858, 4)
Saved bronze/books.parquet -> shape: (271379, 9)
Saved bronze/ratings.parquet -> shape: (1149780, 4)


In [4]:
bronze_users = pd.read_parquet("bronze/users.parquet")
bronze_books = pd.read_parquet("bronze/books.parquet")
bronze_ratings = pd.read_parquet("bronze/ratings.parquet")

tables = {
    "Users": bronze_users,
    "Books": bronze_books,
    "Ratings": bronze_ratings,
}

print("=" * 60)
print("1. SHAPE, DATA TYPES & MISSING VALUES")
print("=" * 60)
for name, df in tables.items():
    print(f"\n--- {name} (Shape: {df.shape}) ---")
    summary = pd.DataFrame({
        "dtype": df.dtypes,
        "null_count": df.isnull().sum(),
        "null_pct": (df.isnull().mean() * 100).round(2)
    })
    print(summary)

print("\n" + "=" * 60)
print("2. RATING DISTRIBUTION")
print("=" * 60)
ratings_num = pd.to_numeric(bronze_ratings["Book-Rating"], errors="coerce")
rating_counts = ratings_num.value_counts(dropna=False).sort_index()
rating_pcts = (ratings_num.value_counts(normalize=True, dropna=False).sort_index() * 100).round(2)
print(pd.DataFrame({"count": rating_counts, "percentage (%)": rating_pcts}))

print("\n" + "=" * 60)
print("3. USER AGES & PUBLICATION YEARS")
print("=" * 60)
ages_num = pd.to_numeric(bronze_users["Age"], errors="coerce")
print("User Age summary (numeric):")
print(ages_num.describe())
print(f"Ages < 5: {(ages_num < 5).sum()} | Ages > 100: {(ages_num > 100).sum()} | Null ages: {ages_num.isnull().sum()}")

years_num = pd.to_numeric(bronze_books["Year-Of-Publication"], errors="coerce")
print("\nPublication Year summary (numeric):")
print(years_num.describe())
print(f"Non-numeric years: {years_num.isnull().sum()} | Year == 0: {(years_num == 0).sum()} | Year > 2026: {(years_num > 2026).sum()}")

print("\n" + "=" * 60)
print("4. RATINGS PER USER & PER BOOK (LONG-TAIL CHECK)")
print("=" * 60)
user_rating_counts = bronze_ratings["User-ID"].value_counts()
book_rating_counts = bronze_ratings["ISBN"].value_counts()

print("Ratings per User quantiles:")
print(user_rating_counts.quantile([0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))
print(f"Users with >= 10 ratings: {(user_rating_counts >= 10).sum()} / {len(user_rating_counts)}")

print("\nRatings per Book quantiles:")
print(book_rating_counts.quantile([0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))
print(f"Books with >= 10 ratings: {(book_rating_counts >= 10).sum()} / {len(book_rating_counts)}")

print("\n" + "=" * 60)
print("5. USER-BOOK MATRIX SPARSITY")
print("=" * 60)
n_unique_users = bronze_ratings["User-ID"].nunique()
n_unique_books = bronze_ratings["ISBN"].nunique()
n_ratings = len(bronze_ratings)
possible_pairs = n_unique_users * n_unique_books
density = (n_ratings / possible_pairs) * 100
sparsity = 100.0 - density

print(f"Unique rated users: {n_unique_users:,}")
print(f"Unique rated books: {n_unique_books:,}")
print(f"Total ratings:      {n_ratings:,}")
print(f"Possible pairs:     {possible_pairs:,}")
print(f"Density:            {density:.6f}% (filled entries)")
print(f"Sparsity:           {sparsity:.6f}% (empty entries)")

1. SHAPE, DATA TYPES & MISSING VALUES

--- Users (Shape: (278858, 4)) ---
                      dtype  null_count  null_pct
User-ID              object           0      0.00
Location             object           0      0.00
Age                  object      110762     39.72
ingestion_timestamp  object           0      0.00

--- Books (Shape: (271379, 9)) ---
                      dtype  null_count  null_pct
ISBN                 object           0       0.0
Book-Title           object           0       0.0
Book-Author          object           2       0.0
Year-Of-Publication  object           0       0.0
Publisher            object           2       0.0
Image-URL-S          object           0       0.0
Image-URL-M          object           0       0.0
Image-URL-L          object           0       0.0
ingestion_timestamp  object           0       0.0

--- Ratings (Shape: (1149780, 4)) ---
                      dtype  null_count  null_pct
User-ID              object           0       0.0
I

### Bronze EDA Findings & Silver Layer Action Plan

1. **Shape, Data Types & Missing Values:** All three tables (`Users`: 278,858 rows, `Books`: 271,379 rows, `Ratings`: 1,149,780 rows) were ingested as raw strings with ~39.7% missing values in `Users.Age` and minor nulls in `Books`; in Silver, I will cast columns to proper numeric/string types, drop unused image URL columns, and impute missing ages.
2. **Distribution of Ratings:** Roughly 62.3% (~716,109 rows) of all ratings are `0` (implicit interactions rather than explicit scores); in Silver, I will filter out `0` ratings and retain only explicit ratings from `1` to `10`.
3. **Distribution of User Ages & Publication Years:** User ages contain impossible outliers (`< 5` and `> 100`, up to 244) and publication years contain `0`, future years (`> 2026`), and shifted text values; in Silver, I will nullify out-of-range ages and impute them with the median age, and set invalid/zero/future publication years to `NaN`.
4. **Ratings per User & per Book:** Both distributions are heavily right-skewed (over 75% of users and books have fewer than 3 ratings); in Silver, I will filter to keep only users and books with at least `10` explicit ratings to eliminate cold-start noise.
5. **Matrix Sparsity:** Only `0.0032%` of all possible user-book pairs contain a rating (`99.9968%` sparse); in Silver and Gold, I will prune low-activity users/books and store the resulting interaction matrix in a memory-efficient SciPy sparse CSR format.

In [6]:
users = pd.read_parquet("bronze/users.parquet")
books = pd.read_parquet("bronze/books.parquet")
ratings = pd.read_parquet("bronze/ratings.parquet")

users_clean = users.drop(columns=["ingestion_timestamp"]).copy()
users_clean["User-ID"] = pd.to_numeric(users_clean["User-ID"], errors="coerce")
users_clean = users_clean.dropna(subset=["User-ID"])
users_clean["User-ID"] = users_clean["User-ID"].astype("int64")

users_clean = users_clean.drop_duplicates(subset=["User-ID"]).copy()

users_clean["Age"] = pd.to_numeric(users_clean["Age"], errors="coerce")
users_clean.loc[(users_clean["Age"] < 5) | (users_clean["Age"] > 100), "Age"] = np.nan
median_age = users_clean["Age"].median()
users_clean["Age"] = users_clean["Age"].fillna(median_age)

loc_split = users_clean["Location"].fillna("").str.split(",")
n_parts = loc_split.str.len()

users_clean["city"] = loc_split.str[0].str.strip().str.title().replace("", np.nan)
users_clean["state"] = np.where(n_parts >= 3, loc_split.str[1].str.strip().str.title(), np.nan)
users_clean["state"] = pd.Series(users_clean["state"], index=users_clean.index).replace("", np.nan)
users_clean["country"] = np.where(n_parts >= 2, loc_split.str[-1].str.strip().str.title(), np.nan)
users_clean["country"] = pd.Series(users_clean["country"], index=users_clean.index).replace("", np.nan)

users_clean = users_clean.drop(columns=["Location"]).rename(columns={
    "User-ID": "user_id",
    "Age": "age"
})

books_clean = books.drop(
    columns=["ingestion_timestamp", "Image-URL-S", "Image-URL-M", "Image-URL-L"],
    errors="ignore"
).copy()

books_clean["ISBN"] = books_clean["ISBN"].astype(str).str.strip().str.upper()
books_clean = books_clean[books_clean["ISBN"].notna() & (books_clean["ISBN"] != "") & (books_clean["ISBN"] != "NAN")]

for col in ["Book-Title", "Book-Author", "Publisher"]:
    books_clean[col] = (
        books_clean[col]
        .fillna("Unknown")
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

books_clean["Book-Author"] = books_clean["Book-Author"].str.title()
books_clean["Publisher"] = books_clean["Publisher"].str.title()

books_clean["Year-Of-Publication"] = pd.to_numeric(books_clean["Year-Of-Publication"], errors="coerce")
books_clean.loc[
    (books_clean["Year-Of-Publication"] <= 0) | (books_clean["Year-Of-Publication"] > 2026),
    "Year-Of-Publication"
] = np.nan

books_clean = books_clean.drop_duplicates(subset=["ISBN"]).rename(columns={
    "ISBN": "isbn",
    "Book-Title": "book_title",
    "Book-Author": "book_author",
    "Year-Of-Publication": "year_of_publication",
    "Publisher": "publisher"
})

ratings_clean = ratings.drop(columns=["ingestion_timestamp"]).copy()
ratings_clean["User-ID"] = pd.to_numeric(ratings_clean["User-ID"], errors="coerce")
ratings_clean["Book-Rating"] = pd.to_numeric(ratings_clean["Book-Rating"], errors="coerce")
ratings_clean["ISBN"] = ratings_clean["ISBN"].astype(str).str.strip().str.upper()

ratings_clean = ratings_clean.dropna(subset=["User-ID", "ISBN", "Book-Rating"]).copy()
ratings_clean["User-ID"] = ratings_clean["User-ID"].astype("int64")
ratings_clean["Book-Rating"] = ratings_clean["Book-Rating"].astype("int64")

ratings_clean = ratings_clean.drop_duplicates(subset=["User-ID", "ISBN"])

ratings_clean = ratings_clean[ratings_clean["Book-Rating"].between(1, 10)].copy()

ratings_clean = ratings_clean.rename(columns={
    "User-ID": "user_id",
    "ISBN": "isbn",
    "Book-Rating": "rating"
})

ratings_clean = ratings_clean[
    ratings_clean["user_id"].isin(users_clean["user_id"]) &
    ratings_clean["isbn"].isin(books_clean["isbn"])
].copy()

MIN_RATINGS = 10

while True:
    prev_len = len(ratings_clean)

    valid_books = ratings_clean["isbn"].value_counts()
    valid_books = valid_books[valid_books >= MIN_RATINGS].index
    ratings_clean = ratings_clean[ratings_clean["isbn"].isin(valid_books)]

    valid_users = ratings_clean["user_id"].value_counts()
    valid_users = valid_users[valid_users >= MIN_RATINGS].index
    ratings_clean = ratings_clean[ratings_clean["user_id"].isin(valid_users)]

    if len(ratings_clean) == prev_len:
        break

users_silver = users_clean[users_clean["user_id"].isin(ratings_clean["user_id"])].reset_index(drop=True)
books_silver = books_clean[books_clean["isbn"].isin(ratings_clean["isbn"])].reset_index(drop=True)
ratings_silver = ratings_clean.reset_index(drop=True)

users_silver.to_parquet("silver/users.parquet", index=False)
books_silver.to_parquet("silver/books.parquet", index=False)
ratings_silver.to_parquet("silver/ratings.parquet", index=False)

print(f"Median Age Imputed: {median_age}")
print(f"Minimum Ratings Threshold: >= {MIN_RATINGS} explicit ratings per user & book")
print(f"Silver Users shape:   {users_silver.shape}")
print(f"Silver Books shape:   {books_silver.shape}")
print(f"Silver Ratings shape: {ratings_silver.shape}")

Median Age Imputed: 32.0
Minimum Ratings Threshold: >= 10 explicit ratings per user & book
Silver Users shape:   (1820, 5)
Silver Books shape:   (2030, 5)
Silver Ratings shape: (41469, 3)


In [7]:
users_silver = pd.read_parquet("silver/users.parquet")
books_silver = pd.read_parquet("silver/books.parquet")
ratings_silver = pd.read_parquet("silver/ratings.parquet")

unique_users = np.sort(ratings_silver["user_id"].unique())
unique_books = np.sort(ratings_silver["isbn"].unique())

user_to_idx = pd.Series(np.arange(len(unique_users)), index=unique_users)
book_to_idx = pd.Series(np.arange(len(unique_books)), index=unique_books)

row_indices = ratings_silver["user_id"].map(user_to_idx).values
col_indices = ratings_silver["isbn"].map(book_to_idx).values
rating_values = ratings_silver["rating"].astype(np.float32).values

user_book_matrix = sparse.csr_matrix(
    (rating_values, (row_indices, col_indices)),
    shape=(len(unique_users), len(unique_books))
)
sparse.save_npz("gold/user_book_matrix.npz", user_book_matrix)

book_lookup = (
    pd.DataFrame({"isbn": unique_books, "book_idx": np.arange(len(unique_books))})
    .merge(books_silver[["isbn", "book_title", "book_author", "year_of_publication", "publisher"]], on="isbn", how="left")
)
book_lookup.to_parquet("gold/book_lookup.parquet", index=False)

user_lookup = (
    pd.DataFrame({"user_id": unique_users, "user_idx": np.arange(len(unique_users))})
    .merge(users_silver[["user_id", "age", "city", "state", "country"]], on="user_id", how="left")
)
user_lookup.to_parquet("gold/user_lookup.parquet", index=False)

user_aggregates = (
    ratings_silver.groupby("user_id", as_index=False)
    .agg(
        rating_count=("rating", "count"),
        avg_rating=("rating", "mean"),
        min_rating=("rating", "min"),
        max_rating=("rating", "max")
    )
    .merge(user_lookup[["user_id", "user_idx"]], on="user_id", how="left")
)
user_aggregates["avg_rating"] = user_aggregates["avg_rating"].round(2)
user_aggregates.to_parquet("gold/user_aggregates.parquet", index=False)

book_aggregates = (
    ratings_silver.groupby("isbn", as_index=False)
    .agg(
        rating_count=("rating", "count"),
        avg_rating=("rating", "mean")
    )
    .merge(book_lookup[["isbn", "book_idx", "book_title", "book_author"]], on="isbn", how="left")
)
book_aggregates["avg_rating"] = book_aggregates["avg_rating"].round(2)
book_aggregates.to_parquet("gold/book_aggregates.parquet", index=False)

print("Gold Layer Artifacts Saved:")
print(f" - gold/user_book_matrix.npz  -> shape: {user_book_matrix.shape}, non-zeros: {user_book_matrix.nnz:,}")
print(f" - gold/book_lookup.parquet   -> shape: {book_lookup.shape}")
print(f" - gold/user_lookup.parquet   -> shape: {user_lookup.shape}")
print(f" - gold/user_aggregates.parquet -> shape: {user_aggregates.shape}")
print(f" - gold/book_aggregates.parquet -> shape: {book_aggregates.shape}")

Gold Layer Artifacts Saved:
 - gold/user_book_matrix.npz  -> shape: (1820, 2030), non-zeros: 41,469
 - gold/book_lookup.parquet   -> shape: (2030, 6)
 - gold/user_lookup.parquet   -> shape: (1820, 6)
 - gold/user_aggregates.parquet -> shape: (1820, 6)
 - gold/book_aggregates.parquet -> shape: (2030, 6)


In [8]:
user_book_matrix = sparse.load_npz("gold/user_book_matrix.npz")
book_lookup = pd.read_parquet("gold/book_lookup.parquet")
user_lookup = pd.read_parquet("gold/user_lookup.parquet")

book_user_matrix = user_book_matrix.T.tocsr()
print(f"Book-User Matrix ready -> shape: {book_user_matrix.shape} (books x users)")

knn_model = NearestNeighbors(metric="cosine", algorithm="brute", n_neighbors=30)
knn_model.fit(book_user_matrix)

user_id_to_idx = dict(zip(user_lookup["user_id"], user_lookup["user_idx"]))
idx_to_book_row = book_lookup.set_index("book_idx")


def get_similar_books(book_title: str, n: int = 10) -> pd.DataFrame:
    title_clean = book_title.strip().lower()

    matches = book_lookup[book_lookup["book_title"].str.lower() == title_clean]
    if matches.empty:
        matches = book_lookup[book_lookup["book_title"].str.lower().str.contains(title_clean, regex=False)]
    if matches.empty:
        raise ValueError(f"Book title '{book_title}' not found in Gold catalog.")

    target_idx = int(matches.iloc[0]["book_idx"])
    matched_title = matches.iloc[0]["book_title"]
    print(f"Query Book: '{matched_title}' (book_idx={target_idx})")

    distances, indices = knn_model.kneighbors(book_user_matrix[target_idx], n_neighbors=n + 1)

    results = []
    for dist, idx in zip(distances.flatten(), indices.flatten()):
        if idx == target_idx:
            continue
        info = idx_to_book_row.loc[idx]
        results.append({
            "book_title": info["book_title"],
            "book_author": info["book_author"],
            "cosine_similarity": round(float(1.0 - dist), 4)
        })
        if len(results) == n:
            break

    return pd.DataFrame(results)


def recommend(user_id: int, n: int = 10) -> pd.DataFrame:
    if user_id not in user_id_to_idx:
        raise ValueError(f"User ID {user_id} not found in Gold user catalog.")

    u_idx = user_id_to_idx[user_id]
    user_row = user_book_matrix[u_idx]

    read_book_indices = set(user_row.indices)
    user_ratings = dict(zip(user_row.indices, user_row.data))

    mean_rating = np.mean(list(user_ratings.values()))
    high_threshold = max(7.0, mean_rating)
    liked_books = [(b_idx, r) for b_idx, r in user_ratings.items() if r >= high_threshold]

    if not liked_books:
        liked_books = sorted(user_ratings.items(), key=lambda x: x[1], reverse=True)[:5]

    candidate_scores = {}
    for b_idx, rating in liked_books:
        distances, indices = knn_model.kneighbors(book_user_matrix[b_idx], n_neighbors=25)
        for dist, neighbor_idx in zip(distances.flatten(), indices.flatten()):
            if neighbor_idx in read_book_indices:
                continue
            similarity = 1.0 - float(dist)
            if similarity > 0:
                candidate_scores[neighbor_idx] = candidate_scores.get(neighbor_idx, 0.0) + (similarity * rating)

    ranked_candidates = sorted(candidate_scores.items(), key=lambda x: x[1], reverse=True)[:n]

    recommendations = []
    for b_idx, score in ranked_candidates:
        info = idx_to_book_row.loc[b_idx]
        recommendations.append({
            "recommended_title": info["book_title"],
            "author": info["book_author"],
            "weighted_score": round(score, 4)
        })

    return pd.DataFrame(recommendations)


print("\nSample Item-to-Item Similarity Check:")
display(get_similar_books("Harry Potter and the Chamber of Secrets (Book 2)", n=10))

Book-User Matrix ready -> shape: (2030, 1820) (books x users)

Sample Item-to-Item Similarity Check:
Query Book: 'Harry Potter and the Chamber of Secrets (Book 2)' (book_idx=863)


,book_title,book_author,cosine_similarity
0,Harry Potter and the Prisoner of Azkaban (Book 3),J. K. Rowling,0.6972
1,Harry Potter and the Sorcerer's Stone (Book 1),J. K. Rowling,0.6689
2,Harry Potter and the Goblet of Fire (Book 4),J. K. Rowling,0.6259
3,Harry Potter and the Order of the Phoenix (Boo...,J. K. Rowling,0.4429
4,The Hobbit: or There and Back Again,J.R.R. Tolkien,0.2445
5,Harry Potter and the Sorcerer's Stone (Harry P...,J. K. Rowling,0.1887
6,The Cat in the Hat,Dr. Seuss,0.1835
7,Harry Potter and the Goblet of Fire (Book 4),J. K. Rowling,0.1805
8,The Fellowship of the Ring (The Lord of the Ri...,J. R. R. Tolkien,0.1743
9,The Bonesetter's Daughter,Amy Tan,0.1721


In [9]:
user_aggs = pd.read_parquet("gold/user_aggregates.parquet")
sample_users = (
    user_aggs[user_aggs["rating_count"].between(15, 60)]
    .sort_values("rating_count", ascending=False)
    .iloc[[0, 25, 75]]["user_id"]
    .tolist()
)

for uid in sample_users:
    print("=" * 90)
    print(f"RECOMMENDATION SANITY CHECK FOR USER ID: {uid}")
    print("=" * 90)

    u_idx = user_id_to_idx[uid]
    u_row = user_book_matrix[u_idx]

    liked_df = pd.DataFrame({
        "book_idx": u_row.indices,
        "user_rating": u_row.data
    }).sort_values("user_rating", ascending=False).head(10)

    liked_df["liked_book_title"] = liked_df["book_idx"].map(idx_to_book_row["book_title"])
    liked_df["liked_author"] = liked_df["book_idx"].map(idx_to_book_row["book_author"])
    liked_df = liked_df[["liked_book_title", "liked_author", "user_rating"]].reset_index(drop=True)

    recs_df = recommend(uid, n=10).reset_index(drop=True)

    comparison_df = pd.concat([liked_df, recs_df], axis=1)
    display(comparison_df)
    print("\n")

RECOMMENDATION SANITY CHECK FOR USER ID: 107784


,liked_book_title,liked_author,user_rating,recommended_title,author,weighted_score
0,The Da Vinci Code,Dan Brown,10.0,Apollyon: The Destroyer Is Unleashed (Left Beh...,Jerry Jenkins,15.7269
1,The Clan of the Cave Bear (Earth's Children (P...,Jean M. Auel,10.0,Nicolae: The Rise of Antichrist (Left Behind N...,Tim F. Lahaye,14.3395
2,The Green Mile,Stephen King,10.0,The Mark: The Beast Rules the World (Left Behi...,Tim Lahaye,11.8279
3,The Poisonwood Bible: A Novel,Barbara Kingsolver,9.0,The Indwelling: The Beast Takes Possession (Le...,Tim Lahaye,11.3147
4,The Testament,John Grisham,9.0,Left Behind: A Novel of the Earth's Last Days ...,Tim Lahaye,11.1915
5,Anne of Green Gables (Anne of Green Gables Nov...,L.M. Montgomery,9.0,The Remnant: On the Brink of Armageddon (Left ...,Tim Lahaye,11.1697
6,The Rainmaker,John Grisham,8.0,"Assassins: Assignment: Jerusalem, Target: Anti...",Tim Lahaye,10.9785
7,Lies and the Lying Liars Who Tell Them: A Fair...,Al Franken,8.0,The Indwelling: The Beast Takes Possession (Le...,Tim Lahaye,10.4929
8,Dances With Wolves,Michael Blake,8.0,The Runaway Jury,John Grisham,9.0902
9,The Firm,John Grisham,8.0,Apollyon: The Destroyer Is Unleashed (Left Beh...,Tim Lahaye,8.2303




RECOMMENDATION SANITY CHECK FOR USER ID: 163804


,liked_book_title,liked_author,user_rating,recommended_title,author,weighted_score
0,Mind Prey,John Sandford,10.0,Three To Get Deadly : A Stephanie Plum Novel (...,Janet Evanovich,14.9187
1,Gone for Good,Harlan Coben,10.0,Hot Six : A Stephanie Plum Novel (A Stephanie ...,Janet Evanovich,13.9620
2,The Lovely Bones: A Novel,Alice Sebold,9.0,High Five (A Stephanie Plum Novel),Janet Evanovich,12.9801
3,One for the Money (A Stephanie Plum Novel),Janet Evanovich,9.0,Seven Up (A Stephanie Plum Novel),Janet Evanovich,11.4926
4,Chosen Prey,John Sandford,9.0,Winter Prey,John Sandford,10.0730
5,Rules of Prey,John Sandford,9.0,Eyes of Prey,John Sandford,9.7459
6,Mortal Prey,John Sandford,9.0,"Cerulean Sins: An Anita Blake, Vampire Hunter ...",Laurell K. Hamilton,9.0919
7,The Green Mile: Coffey's Hands (Green Mile Ser...,Stephen King,9.0,Shadow Prey,John Sandford,8.9318
8,The Green Mile: The Bad Death of Eduard Delacr...,Stephen King,9.0,Animal Farm,George Orwell,8.8974
9,The Green Mile: Coffey on the Mile (Green Mile...,Stephen King,9.0,To the Nines: A Stephanie Plum Novel,Janet Evanovich,8.8667




RECOMMENDATION SANITY CHECK FOR USER ID: 130571


,liked_book_title,liked_author,user_rating,recommended_title,author,weighted_score
0,Bel Canto: A Novel,Ann Patchett,9.0,E Is for Evidence: A Kinsey Millhone Mystery (...,Sue Grafton,9.2333
1,I Capture the Castle,Dodie Smith,9.0,The Clinic (Alex Delaware Novels (Paperback)),Jonathan Kellerman,8.3141
2,Plainsong (Vintage Contemporaries),Kent Haruf,9.0,Time Bomb (Alex Delaware Novels (Paperback)),Jonathan Kellerman,7.6887
3,Into the Wild,Jon Krakauer,9.0,H Is for Homicide (Kinsey Millhone Mysteries (...,Sue Grafton,7.6490
4,Seinlanguage,Jerry Seinfeld,9.0,J Is for Judgment,Sue Grafton,7.5526
5,Missing Joseph,Elizabeth George,9.0,G Is for Gumshoe (Kinsey Millhone Mysteries (P...,Sue Grafton,7.3972
6,Peace Like a River,Leif Enger,9.0,Ill Wind (Anna Pigeon Mysteries (Paperback)),Nevada Barr,7.3467
7,Mystic River,Dennis Lehane,9.0,D Is for Deadbeat (Kinsey Millhone Mysteries (...,Sue Grafton,6.9264
8,Eyes of Prey,John Sandford,9.0,Firestorm (Anna Pigeon Mysteries (Paperback)),Nevada Barr,6.4654
9,Where the Heart Is (Oprah's Book Club (Paperba...,Billie Letts,9.0,M Is for Malice,Sue Grafton,6.3431
